# Lab 12: LSTM Model for Time-Series Forecasting

**Student Name:** Samiullah Khan  
**Registration Number:** 22JZELE0492  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus  

## Lab Overview
This notebook develops an LSTM-based deep learning model for time-series forecasting. It imports forecasting utilities, defines an LSTM architecture, configures callbacks and checkpoints, loads train/validation/test datasets, trains the model, and evaluates forecasting performance using regression metrics.

## Learning Objectives
- Set the working directory and import Scikit-learn, TensorFlow/Keras, and custom time-series utilities.
- Define an LSTM neural network architecture for sequential forecasting data.
- Configure optimizer, model checkpoints, and training-monitor callbacks.
- Load training, validation, and testing CSV files for time-series prediction.
- Evaluate LSTM forecasting output using MAE, MSE, RMSE, R2 score, and explained variance.

## Notebook Format
The original code logic and existing workflow are preserved. Additional headings and comments are used only to make the notebook more professional and easier to follow.

## Section 1: Working Directory and Library Imports
This section sets the Lab 12 working directory and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utility functions.


In [2]:
# Set the working directory for Lab 12 files.
import os
os.chdir(r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12')

In [3]:
# Import metrics, time-series helpers, callbacks, TensorFlow/Keras layers, and utilities.
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [4]:
# Define training state and input dimensions for the time-series windows.
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [5]:
# Build an LSTM model for sequence-based forecasting.
def create_lstm():
    input_data = Input(shape=(time_steps, num_features))
    lstm_layer1 = LSTM(8, return_sequences=True)(input_data)
    lstm_layer2 = LSTM(20)(lstm_layer1)
    x = Flatten()(lstm_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

## Section 2: Model Parameters and LSTM Architecture
The following cells define time steps, feature count, and the LSTM model architecture used for sequential forecasting.


In [6]:
model1 = create_lstm()
model1.summary()

Model: "functional"

â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”³â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”³â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”“
â”ƒ Layer (type)                    â”ƒ Output Shape           â”ƒ       Param # â”ƒ
â”¡â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â•‡â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â•‡â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”©
â”‚ input_layer (InputLayer)        â”‚ (None, 24, 21)         â”‚             0 â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ lstm (LSTM)                     â”‚ (None, 24, 8)          â”‚           960 â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ lstm_1 (LSTM)                   â”‚ (None, 20)             â”‚         2,320 â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ flatten (Flatten)               â”‚ (None, 20)             â”‚             0 â”‚
â”œâ”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¼â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”¤
â”‚ dense (Dense)                   â”‚ (None, 1)              â”‚            21 â”‚
â””â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”´â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”´â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”˜

 Total params: 3,301 (12.89 KB)

 Trainable params: 3,301 (12.89 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [8]:
# Define checkpoint, output, figure, and training-history paths.
checkpoints = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [9]:
# Save only the best model checkpoint based on validation loss.
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# Group callbacks for model training.
callbacks = [EpochCheckpoint1,TrainingMonitor1]

## Section 3: Checkpoints, Callbacks, and Training Configuration
This section prepares checkpoint paths, output directories, training monitor callbacks, optimizer settings, and model compilation.


In [10]:
# Compile a new LSTM model or load an existing checkpoint for continued training.
if model is None:
    print("[INFO] compiling model...")
    model =create_lstm()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # Update the learning rate before resuming training.
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [11]:
# Load train, validation, test, and original hourly CSV files.
path_dataset = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12'

path_tr = os.path.join(path_dataset, 'AEP_train.csv')
path_v = os.path.join(path_dataset, 'AEP_validation.csv')
path_te = os.path.join(path_dataset, 'AEP_test.csv')
path_hourly = os.path.join(path_dataset, 'AEP_hourly.csv')

for path in [path_tr, path_v, path_te, path_hourly]:
    print(path, "=>", os.path.exists(path))

df_tr = pd.read_csv(path_tr)
train_set = df_tr.values

df_v = pd.read_csv(path_v)
validation_set = df_v.values

df_te = pd.read_csv(path_te)
test_set = df_te.values

df_hourly = pd.read_csv(path_hourly)

train_set.shape, validation_set.shape, test_set.shape, df_hourly.shape

C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\AEP_train.csv => True
C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\AEP_validation.csv => True
C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\AEP_test.csv => True
C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\AEP_hourly.csv => True


((84907, 21), (24259, 21), (12130, 21), (121273, 2))

In [12]:
# Confirm the forecasting window size and feature count.
time_steps=24
num_features=21

In [13]:
# Convert raw arrays into univariate multi-step forecasting windows.
start = time.time()

train_X, train_y = univariate_multi_step(
    train_set,
    time_steps,
    target_col=0,
    target_len=1
)

validation_X, validation_y = univariate_multi_step(
    validation_set,
    time_steps,
    target_col=0,
    target_len=1
)

test_X, test_y = univariate_multi_step(
    test_set,
    time_steps,
    target_col=0,
    target_len=1
)

print('Time Consumed', time.time() - start, "sec")

Time Consumed 0.7943837642669678 sec


## Section 4: Dataset Loading and Forecast Preparation
The following cells load train, validation, and test CSV files, then prepare the data for LSTM forecasting input/output format.


In [14]:
# Train the LSTM model on the prepared forecasting windows.
epochs = 1
verbose = 1
batch_size = 32

History = model.fit(
    train_X,
    train_y,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(validation_X, validation_y),
    callbacks=callbacks,
    verbose=verbose
)

2650/2653 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 20ms/step - loss: 0.0753 - mae: 0.0753 - mape: 469.1146
Epoch 1: val_loss improved from None to 0.02279, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\\E1-cp-0001-loss0.02.h5


2653/2653 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 65s 23ms/step - loss: 0.0474 - mae: 0.0474 - mape: 528.0041 - val_loss: 0.0228 - val_mae: 0.0228 - val_mape: 9.7585


In [16]:
# Load the fitted scaler used to inverse-transform predictions.
import os
import pickle

path_dataset = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12'
path_scaler = os.path.join(path_dataset, 'AEP_Scaler.pkl')

print(os.path.exists(path_scaler))

with open(path_scaler, 'rb') as f:
    scaler = pickle.load(f)

True


c:\Users\Sami\anaconda3\envs\machinelearning\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [17]:
# Load the trained model, predict the test set, inverse-transform values, and evaluate metrics.
model_path = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\\E1-cp-0001-loss0.02.h5'

model = load_model(model_path, compile=False)

y_pred_scaled = model.predict(test_X)

print("y_pred_scaled.shape =", y_pred_scaled.shape)
print("test_y.shape =", test_y.shape)

def inverse_target(scaled_values, scaler, target_col=0, num_features=21):
    scaled_values = scaled_values.reshape(-1)

    dummy = np.zeros((scaled_values.shape[0], num_features))
    dummy[:, target_col] = scaled_values

    unscaled = scaler.inverse_transform(dummy)

    return unscaled[:, target_col].reshape(-1, 1)

y_pred = inverse_target(
    y_pred_scaled,
    scaler,
    target_col=0,
    num_features=21
)

y_test_unscaled = inverse_target(
    test_y,
    scaler,
    target_col=0,
    num_features=21
)

# Mean Absolute Error (MAE)
MAE = np.mean(np.abs(y_pred - y_test_unscaled))
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(np.abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.mean(np.square(y_pred - y_test_unscaled))
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squared Error (RMSE)
RMSE = np.sqrt(MSE)
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape = ', y_test_unscaled.shape)
print('y_pred.shape = ', y_pred.shape)

379/379 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 4s 9ms/step
y_pred_scaled.shape = (12105, 1)
test_y.shape = (12105, 1)
Mean Absolute Error (MAE): 365.75
Median Absolute Error (MedAE): 301.51
Mean Squared Error (MSE): 217840.28
Root Mean Squared Error (RMSE): 466.73
Mean Absolute Percentage Error (MAPE): 2.49 %
Median Absolute Percentage Error (MDAPE): 2.09 %


y_test_unscaled.shape =  (12105, 1)
y_pred.shape =  (12105, 1)


In [18]:
# Configure fine-tuning checkpoint path and starting model.
checkpoints = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\\E1-cp-0001-loss0.02.h5'
start_epoch= 34

## Section 5: Prediction and Model Evaluation
The final cells generate predictions and calculate regression metrics to evaluate the forecasting model performance.


In [20]:
# Set up fine-tuning callbacks and load the previous LSTM checkpoint.
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# Group callbacks for continued training.
callbacks = [EpochCheckpoint1,TrainingMonitor1]
model_load = r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\\E1-cp-0001-loss0.02.h5'
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
else:
    print("[INFO] loading {}...".format(model_load))
    model = load_model(model_load, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )


    # Display the learning rate used for fine tuning.
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\\E1-cp-0001-loss0.02.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


In [21]:
# Continue training the loaded LSTM model.
epochs = 2
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/2
2652/2653 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 18ms/step - loss: 0.0209 - mae: 0.0209 - mape: 42.1428
Epoch 1: val_loss improved from None to 0.01922, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\E2-cp-0001-loss0.02.h5


2653/2653 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 58s 20ms/step - loss: 0.0203 - mae: 0.0203 - mape: 35.0897 - val_loss: 0.0192 - val_mae: 0.0192 - val_mape: 8.2274
Epoch 2/2
2650/2653 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 0s 18ms/step - loss: 0.0190 - mae: 0.0190 - mape: 126.1915
Epoch 2: val_loss improved from 0.01922 to 0.01729, saving model to C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\E2-cp-0002-loss0.02.h5


2653/2653 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 84s 21ms/step - loss: 0.0185 - mae: 0.0185 - mape: 296.5967 - val_loss: 0.0173 - val_mae: 0.0173 - val_mape: 7.3125


In [22]:
# Load the fine-tuned model and evaluate forecasting performance on the test set.
model = load_model(r'C:\Users\Sami\Documents\MachineLearning\Material\ML\Lab 12\E2-cp-0002-loss0.02.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squared Error (RMSE)
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â”â” 3s 8ms/step
Mean Absolute Error (MAE): 278.81
Median Absolute Error (MedAE): 229.5
Mean Squared Error (MSE): 127693.6
Root Mean Squared Error (RMSE): 357.34
Mean Absolute Percentage Error (MAPE): 1.9 %
Median Absolute Percentage Error (MDAPE): 1.59 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## Final Conclusion
In this lab, an LSTM neural network was implemented for time-series forecasting. The notebook demonstrates sequence-model architecture, callback-based training, data preparation, fine tuning, and regression-based model evaluation.

## Submitted By
**Student Name:** Samiullah Khan  
**Registration Number:** 22JZELE0492  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus